In [7]:
import os

parent_dir = "/data3/jhpark/data_synth/celeba-hq"  # Change this to your directory path

# Collect all image file paths (without loading images into memory)
image_extensions = ('.png', '.jpg', '.jpeg', '.webp')
image_paths = []
for root, dirs, files in os.walk(parent_dir):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images in {parent_dir}")


Found 28000 images in /data3/jhpark/data_synth/celeba-hq


In [8]:
import shutil
import math

def split_images_into_subdirectories(image_paths, output_dir, k=100):
    """
    Split images into subdirectories with maximum k images per subdirectory
    
    Args:
        image_paths: List of image file paths
        output_dir: Directory where subdirectories will be created
        k: Maximum number of images per subdirectory
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Calculate number of subdirectories needed
    num_subdirs = math.ceil(len(image_paths) / k)
    
    print(f"Total images: {len(image_paths)}")
    print(f"Images per subdirectory (k): {k}")
    print(f"Number of subdirectories to create: {num_subdirs}")
    
    # Split images into subdirectories
    for i in range(num_subdirs):
        # Create subdirectory
        subdir_name = f"batch_{i+1:03d}"  # e.g., batch_001, batch_002, etc.
        subdir_path = os.path.join(output_dir, subdir_name)
        os.makedirs(subdir_path, exist_ok=True)
        
        # Get images for this subdirectory
        start_idx = i * k
        end_idx = min((i + 1) * k, len(image_paths))
        batch_images = image_paths[start_idx:end_idx]
        
        print(f"Creating {subdir_name} with {len(batch_images)} images")
        
        # Copy images to subdirectory
        for img_path in batch_images:
            img_filename = os.path.basename(img_path)
            dest_path = os.path.join(subdir_path, img_filename)
            
            # Copy the file (preserves original)
            shutil.copy2(img_path, dest_path)
    
    print("Dataset splitting completed!")

# Set parameters
k = 10000  # Maximum images per subdirectory (you can change this)
output_directory = "/data3/jhpark/data_synth/celeba-hq_split"  # Output directory

# Split the images
split_images_into_subdirectories(image_paths, output_directory, k)


Total images: 28000
Images per subdirectory (k): 10000
Number of subdirectories to create: 3
Creating batch_001 with 10000 images
Creating batch_002 with 10000 images
Creating batch_003 with 8000 images
Dataset splitting completed!


In [ ]:
# Alternative function to move files instead of copying them
def split_images_move(image_paths, output_dir, k=100):
    """
    Split images into subdirectories by moving files (faster but removes originals)
    """
    os.makedirs(output_dir, exist_ok=True)
    num_subdirs = math.ceil(len(image_paths) / k)
    
    print(f"Moving {len(image_paths)} images into {num_subdirs} subdirectories")
    
    for i in range(num_subdirs):
        subdir_name = f"batch_{i+1:03d}"
        subdir_path = os.path.join(output_dir, subdir_name)
        os.makedirs(subdir_path, exist_ok=True)
        
        start_idx = i * k
        end_idx = min((i + 1) * k, len(image_paths))
        batch_images = image_paths[start_idx:end_idx]
        
        for img_path in batch_images:
            img_filename = os.path.basename(img_path)
            dest_path = os.path.join(subdir_path, img_filename)
            
            # Handle duplicate filenames by adding a counter
            counter = 1
            base_name, ext = os.path.splitext(img_filename)
            while os.path.exists(dest_path):
                img_filename = f"{base_name}_{counter:03d}{ext}"
                dest_path = os.path.join(subdir_path, img_filename)
                counter += 1
            
            shutil.move(img_path, dest_path)
    
    print("Dataset splitting (move) completed!")

# Uncomment the line below to use move instead of copy
# split_images_move(image_paths, output_directory, k)
